# Day 078 — Exercise 3: transcribe_media

**What you'll build:** The audio transcription pipeline — accepts bytes or file path, returns a normalised transcript dict.

**Why it matters:** `transcribe_media` brings the Day 72 Whisper integration into the studio. The dual input form handles both uploaded audio bytes and on-disk audio files.

In [ ]:
from PIL import Image as _PILImage

def _make_mock_image(w=100, h=100, color=(100, 150, 200)):
    return _PILImage.new('RGB', (w, h), color=color)

_mock_describe_fn   = lambda img, q: 'A test image with a solid color background.'
_mock_transcribe_fn = lambda src: {'text': 'Hello world.', 'segments': [
    {'start': 0.0, 'end': 1.0, 'text': 'Hello world.'}]}
_mock_tts_fn        = lambda text, voice, rate, pitch: b'AUDIO:' + text[:8].encode()
from pathlib import Path

MEDIA_EXTENSIONS = {
    'image': {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tiff'},
    'audio': {'.mp3', '.wav', '.ogg', '.flac', '.m4a', '.aac'},
    'video': {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv'},
}

def detect_media_type(path):
    ext = Path(path).suffix.lower()
    for media_type, extensions in MEDIA_EXTENSIONS.items():
        if ext in extensions:
            return media_type
    return 'unknown'
import io, base64

def describe_media(source, describe_fn=None):
    from PIL import Image
    if isinstance(source, (str, Path)):
        image = Image.open(source)
    else:
        image = source
    prompt = 'Describe this image in detail, including all visible content.'
    if describe_fn is not None:
        return describe_fn(image, prompt)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}],
    )
    return resp['message']['content']


## Task

`transcribe_media(source, transcribe_fn=None) -> dict`

1. If `transcribe_fn`: `return transcribe_fn(source)`
2. `import whisper, os, tempfile; model = whisper.load_model('base')`
3. If `isinstance(source, bytes)`: write to `NamedTemporaryFile(suffix='.wav', delete=False)`, `try: raw=model.transcribe(tmp); finally: os.unlink(tmp)`
4. Else: `raw = model.transcribe(str(source))`
5. `segments = [{'start':s['start'],'end':s['end'],'text':s['text'].strip()} for s in raw.get('segments',[])]`
6. Return `{'text': raw.get('text','').strip(), 'segments': segments}`

## Your Implementation

In [ ]:
def transcribe_media(source, transcribe_fn=None):
    """Transcribe audio (bytes or path) using Whisper.
    Returns dict with keys: text (str), segments (list).
    """
    raise NotImplementedError


In [ ]:
def transcribe_media(source, transcribe_fn=None):
    if transcribe_fn is not None:
        return transcribe_fn(source)
    import whisper, os, tempfile
    model = whisper.load_model('base')
    if isinstance(source, bytes):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(source)
            tmp = f.name
        try:
            raw = model.transcribe(tmp)
        finally:
            os.unlink(tmp)
    else:
        raw = model.transcribe(str(source))
    segments = [
        {'start': s['start'], 'end': s['end'], 'text': s['text'].strip()}
        for s in raw.get('segments', [])
    ]
    return {'text': raw.get('text', '').strip(), 'segments': segments}


## Automated checks

In [ ]:

score, total = 0, 4
try:
    result = transcribe_media(b'MOCK_AUDIO', transcribe_fn=_mock_transcribe_fn)
    assert isinstance(result, dict)
    score += 1; print("✅ transcribe_media returns dict")

    assert 'text' in result and 'segments' in result
    score += 1; print("✅ result has 'text' and 'segments' keys")

    assert isinstance(result['text'], str)
    assert isinstance(result['segments'], list)
    score += 1; print("✅ text is str, segments is list")

    called = []
    def _tfn(src): called.append(src); return {'text': 'hi', 'segments': []}
    transcribe_media('test.wav', transcribe_fn=_tfn)
    assert called[0] == 'test.wav', f"transcribe_fn not called with source: {called}"
    score += 1; print("✅ transcribe_fn receives source directly")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def transcribe_media(source, transcribe_fn=None):
    if transcribe_fn is not None:
        return transcribe_fn(source)
    import whisper, os, tempfile
    model = whisper.load_model('base')
    if isinstance(source, bytes):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(source)
            tmp = f.name
        try:
            raw = model.transcribe(tmp)
        finally:
            os.unlink(tmp)
    else:
        raw = model.transcribe(str(source))
    segments = [
        {'start': s['start'], 'end': s['end'], 'text': s['text'].strip()}
        for s in raw.get('segments', [])
    ]
    return {'text': raw.get('text', '').strip(), 'segments': segments}
```

**Why `transcribe_fn(source)` not `transcribe_fn(source, model)`?** The mock doesn't need the model — it returns a fixed dict. The injection replaces the entire transcription operation, not just one part of it.

</details>